# NutriWise — Tier-2 fallback: Open Food Facts (OFF)

When BLS has **no** entry for a receipt line, we fall back to **Open Food Facts** — a
crowd-sourced, barcode-based product database with huge branded coverage. This mirrors the
product's tiered design: **BLS (reliable German nutrition) → OFF (branded coverage) → learned
verified matches**.

**Scope.** Only the `no_bls_match` items get an OFF lookup. Of the 6 such line items, 4 are
recovered here; the ambiguous counter fish (`Frischfisch`, species unspecified) stays
unresolved — that is an *ambiguity*, not a database gap, so we do not guess.

**Honesty guardrails.**
- OFF nutrient data is crowd-sourced (variable quality) — great for *identifying* a product,
  weaker for *precise nutrition* than BLS. We therefore tag each label with `gold_source`
  (`bls` vs `off`) and never mix the two silently.
- OFF matches were judged (same discipline as the BLS gold) and get a small spot-check.

---
## How we queried OFF (transparent, free-text endpoint)

In [1]:
import json, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()

def off_search(terms, page_size=5, timeout=15):
    """Free-text search against the German OFF instance. Returns list of (code, name, brands).
    Network-dependent; used here only to *illustrate* the lookup (results cached below)."""
    base = "https://de.openfoodfacts.org/cgi/search.pl"
    q = urllib.parse.urlencode({
        "search_terms": terms, "search_simple": 1, "action": "process",
        "json": 1, "page_size": page_size,
        "fields": "product_name,brands,code",
    })
    req = urllib.request.Request(base + "?" + q, headers={"User-Agent": "NutriWise-capstone/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        d = json.loads(r.read().decode("utf-8"))
    return [(p.get("code"), (p.get("product_name") or "").strip(), p.get("brands"))
            for p in d.get("products", [])]

# Optional live demo (safe if offline — cached results below are authoritative)
try:
    demo = off_search("Airwaves Kaugummi", page_size=3)
    print("Live OFF sample for 'Airwaves Kaugummi':")
    for c, n, b in demo: print(f"  [{c}] {n!r} | {b}")
except Exception as e:
    print("(Live query skipped:", e, ")")

(Live query skipped: HTTP Error 503: Service Temporarily Unavailable )


---
## Judged OFF matches (cached, authoritative)

Queried live on 2026-07-17, then judged. Kept as data so this notebook is reproducible
offline and the judgments are auditable.

In [2]:
# query_standard -> resolved & judged OFF match
OFF_MATCHES = {
    "Airwaves Cool Cassis Kaugummi (50 Stück)": {
        "off_code": "4009900537698", "off_name": "Airwaves Cool Cassis", "off_brand": "Airwaves",
        "verdict": "correct", "reasoning": "Exact branded match (same product)."},
    "Ostmann Oregano gerebelt 25g": {
        "off_code": "4002674073959", "off_name": "Oregano gerebelt", "off_brand": "Ostmann",
        "verdict": "correct", "reasoning": "Exact branded match (Ostmann rubbed oregano)."},
    "Ostmann Paprika edelsüß gemahlen 50g": {
        "off_code": "4002674054071", "off_name": "Paprika edelsüß gemahlen",
        "off_brand": "Ostmann (EDEKA/Fuchs)",
        "verdict": "correct", "reasoning": "Exact branded match (Ostmann ground sweet paprika)."},
    "Ostmann Rosmarin geschnitten 50g": {
        "off_code": "9001414024690", "off_name": "Rosmarin geschnitten, getrocknet",
        "off_brand": "Kotányi",
        "verdict": "partially_correct",
        "reasoning": "Same product form (cut dried rosemary); different brand (no Ostmann entry in OFF)."},
}
for q, m in OFF_MATCHES.items():
    print(f"{q}\n   -> [{m['off_code']}] {m['off_name']} ({m['off_brand']}) [{m['verdict']}]")

Airwaves Cool Cassis Kaugummi (50 Stück)
   -> [4009900537698] Airwaves Cool Cassis (Airwaves) [correct]
Ostmann Oregano gerebelt 25g
   -> [4002674073959] Oregano gerebelt (Ostmann) [correct]
Ostmann Paprika edelsüß gemahlen 50g
   -> [4002674054071] Paprika edelsüß gemahlen (Ostmann (EDEKA/Fuchs)) [correct]
Ostmann Rosmarin geschnitten 50g
   -> [9001414024690] Rosmarin geschnitten, getrocknet (Kotányi) [partially_correct]


---
## Apply to the gold-set (tagging `gold_source`)

In [3]:
QUERIES = REPO / "receipts" / "receipts_queries.json"
recs = json.loads(QUERIES.read_text(encoding="utf-8"))

applied = 0
for r in recs:
    # tag existing BLS labels for clean separation
    if r["verdict"] in ("correct", "partially_correct") and r.get("gold_corpus_idx") is not None:
        r["gold_source"] = "bls"
    m = OFF_MATCHES.get(r["name_standard"])
    if m and r["verdict"] == "no_bls_match":
        r["gold_source"] = "off"
        r["verdict"] = m["verdict"]
        r["off_code"] = m["off_code"]
        r["off_matched_name"] = m["off_name"]
        r["gold_bls_name"] = None          # not a BLS entry
        r["gold_corpus_idx"] = None
        r["label_source"] = "off_fallback+judge/2026-07-17"
        r["reasoning"] = m["reasoning"]
        applied += 1

QUERIES.write_text(json.dumps(recs, ensure_ascii=False, indent=2), encoding="utf-8")

from collections import Counter
usable = [r for r in recs if r["verdict"] in ("correct", "partially_correct")]
src = Counter(r.get("gold_source") for r in usable)
print(f"OFF labels applied to line items: {applied}")
print(f"Usable gold now: {len(usable)}/180")
print(f"  by source: BLS={src.get('bls',0)}, OFF={src.get('off',0)}")
print(f"  still no_bls_match: {sum(1 for r in recs if r['verdict']=='no_bls_match')} "
      f"(ambiguous counter fish — kept honest)")

OFF labels applied to line items: 4
Usable gold now: 178/180
  by source: BLS=174, OFF=4
  still no_bls_match: 2 (ambiguous counter fish — kept honest)


---
## Result

- **Coverage 174 → 178 / 180.** The 4 recovered items are exactly the branded/spice products
  BLS structurally lacks — OFF's sweet spot.
- **The tiered design works and is honest:** BLS for reliable German nutrition, OFF for
  branded coverage, each tagged by `gold_source`. Nutrient quality is not mixed silently.
- **Two line items stay unresolved** (`Frischfisch`, species unspecified) — an ambiguity we
  refuse to guess, consistent with "honesty as a feature".
- **Not in the BLS-corpus ranking eval:** OFF items live in a different corpus, so they are a
  *coverage* win, reported separately from the BLS ExpTop-1 numbers in `goldset_eval.ipynb`.

**Next:** a 4-item spot-check of these OFF matches (`ml/spotcheck_off.json`) to fold into the
label-error rate.